# Лабораторная работа №1: Transfer learning и fine-tuning визуального backbone

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
- Сравнить frozen, partial и full fine-tuning ResNet18.
- Построить confusion matrix и проверить corruption shift.


## Теоретическая часть

Предобученный backbone реализует отображение $f_\theta(x)=z$. Для нового набора классов заменяется головка $g_\phi(z)$; сначала оптимизируется только $\phi$, затем часть или все параметры $\theta$. Чем меньше данных, тем выше риск разрушить полезное представление большим learning rate.

![Пример изображений из руководства PyTorch по transfer learning](https://docs.pytorch.org/tutorials/_images/sphx_glr_transfer_learning_tutorial_001.png)

Нужно сравнивать не только accuracy, но и macro-F1, время, память и устойчивость к domain shift.


### Математическая постановка

Для многоклассовой задачи используется cross-entropy:
$$\mathcal L=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid x_i).$$
Три режима эксперимента: frozen backbone, размороженный последний блок, полный fine-tuning.


### Материалы

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision models](https://pytorch.org/vision/stable/models.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
2. Сравнить frozen, partial и full fine-tuning ResNet18.
3. Построить confusion matrix и проверить corruption shift.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** ResNet18, FP32/FP16, 32–224 px.
- **Ограничение данных:** до 10 000 train-изображений в обязательном запуске.
- **Fallback:** CPU/FAST_DEV_RUN на подвыборке.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [ ]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# pin_memory полезен только на CUDA — на CPU он вызывает UserWarning
PIN_MEMORY = (DEVICE == "cuda")

print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT), "pin_memory": PIN_MEMORY})

{'device': 'cpu', 'fast_dev_run': False, 'data_root': '/data', 'pin_memory': False}


In [ ]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Разделение CIFAR-10

*Вес: 1 балл.*

**Постановка.** Изоляция test необходима для оценки обобщения без подбора по тестовым меткам.

Выполните следующие действия:
1. Загрузите CIFAR-10 и разделите train на стратифицированные train и validation.
2. Подготовьте DataLoader с детерминированным test-преобразованием.

**Результат.** Непересекающиеся train, validation и test и загрузчики train_loader, val_loader, test_loader.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Стратифицированное разделение CIFAR-10: train/val без пересечений,
# test полностью изолирован до финального замера.
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

# Русские названия классов CIFAR-10 (индексы совпадают с метками датасета)
CLASS_NAMES_RU = [
    "самолёт", "автомобиль", "птица", "кошка", "олень",
    "собака", "лягушка", "лошадь", "корабль", "грузовик",
]

# Нормализация CIFAR-10 (средние/дисперсии по каналам)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
# Детерминированное test/val-преобразование
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_full = datasets.CIFAR10(DATA_ROOT, train=True,  download=True, transform=train_tf)
train_eval = datasets.CIFAR10(DATA_ROOT, train=True,  download=True, transform=eval_tf)
test_set   = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=eval_tf)

targets = np.array(train_full.targets)
idx = np.arange(len(targets))
train_idx, val_idx = train_test_split(
    idx, test_size=0.1, stratify=targets, random_state=SEED
)

# Ограничение по профилю вычислений
if FAST_DEV_RUN:
    train_idx = train_idx[:500]
    val_idx   = val_idx[:200]

train_set = Subset(train_full, train_idx)
val_set   = Subset(train_eval, val_idx)

batch_size = 32 if FAST_DEV_RUN else 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=PIN_MEMORY)

# Проверки
assert set(train_idx).isdisjoint(set(val_idx)), "Утечка train↔val"
check(len(train_idx) > 0 and len(val_idx) > 0,
      f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_set)}")
print("Классы:", CLASS_NAMES_RU)

✔ train=45000, val=5000, test=10000
Классы: ['самолёт', 'автомобиль', 'птица', 'кошка', 'олень', 'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик']


Разделили 50 000 train на 45 000 train и 5 000 validation, test оставили нетронутым — 10 000 изображений. Разделение стратифицированное, классы в train и val распределены одинаково. Val и test используют одну и ту же детерминированную трансформацию без аугментаций, а train — с RandomCrop и RandomHorizontalFlip. Пересечений между train и val нет, test полностью изолирован до финального замера.

### Backbone и функции обучения

*Вес: 1 балл.*

**Постановка.** Режимы заморозки изменяют множество обновляемых параметров. Сравнивать их можно лишь при одинаковой функции потерь.

Выполните следующие действия:
1. Создайте ResNet18 с новой десятиклассовой головой.
2. Определите функции обучения и вычисления accuracy, macro-F1, матрицы ошибок.

**Результат.** Функции build_model, train_epoch и evaluate.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# ResNet18 с заменённой головкой; функции эпохи и оценки метрик
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import f1_score, confusion_matrix


def build_model(mode: str = "full", num_classes: int = 10):
    """mode ∈ {'frozen','partial','full'}: какие параметры обучаются."""
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    if mode == "frozen":
        # обучаем только новую головку
        for p in model.parameters():
            p.requires_grad = False
        for p in model.fc.parameters():
            p.requires_grad = True
    elif mode == "partial":
        # размораживаем последний residual-блок + головку
        for p in model.parameters():
            p.requires_grad = False
        for p in model.layer4.parameters():
            p.requires_grad = True
        for p in model.fc.parameters():
            p.requires_grad = True
    elif mode == "full":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Неизвестный режим: {mode}")
    return model


def train_epoch(model, loader, optim, criterion, device):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optim.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optim.step()
        loss_sum += loss.item() * x.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        total    += x.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    ys, ps, loss_sum, total = [], [], 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss_sum += crit(out, y).item() * x.size(0)
        ys.append(y.cpu().numpy())
        ps.append(out.argmax(1).cpu().numpy())
        total += x.size(0)
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    return {
        "loss": loss_sum / total,
        "acc": float((y_true == y_pred).mean()),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "confusion": confusion_matrix(y_true, y_pred).tolist(),
        "y_true": y_true, "y_pred": y_pred,
    }


def jsonable(metrics: dict) -> dict:
    """Убираем numpy-массивы перед записью в JSON."""
    return {k: v for k, v in metrics.items() if k not in ("y_true", "y_pred")}

Собрали ResNet18 с новой головой под 10 классов. Функция build_model умеет включать три режима заморозки через requires_grad. train_epoch и evaluate считают loss, accuracy, macro-F1 и confusion matrix — одни и те же для всех режимов. Благодаря этому сравнение получится честным: различие между режимами только в том, какие веса обновляются.

### Режимы fine-tuning

*Вес: 1 балл.*

**Постановка.** Валидация определяет режим разморозки до первого обращения к test.

Выполните следующие действия:
1. Обучите frozen, partial и full при одинаковом числе эпох.
2. Сохраните веса и метрики validation; выберите режим по macro-F1.

**Результат.** Файлы resnet18_*.pt и metrics.json с измеренными validation-метриками.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


frozen — обучается только головка (5 130 параметров). Backbone не меняется.

partial — разморожен layer4 (последний residual-блок) + головка (8.4M параметров).

full — обучаются все параметры (11.2M).

In [ ]:
# Режимы fine-tuning: frozen / partial / full, ровно 3 эпохи для каждого.
# Валидация определяет победителя до первого обращения к test.

# --- 1. Ограничение train до 10 000 объектов (профиль вычислений) ---
MAX_TRAIN = 10000
if len(train_set) > MAX_TRAIN:
    sub_idx = np.random.RandomState(SEED).permutation(len(train_set))[:MAX_TRAIN]
    train_set_limited = Subset(train_set.dataset, [train_set.indices[i] for i in sub_idx])
else:
    train_set_limited = train_set

train_loader_limited = DataLoader(
    train_set_limited, batch_size=batch_size, shuffle=True,
    num_workers=2, pin_memory=PIN_MEMORY,
)

print(f"Train для fine-tuning: {len(train_set_limited)} объектов")

# --- 2. Жёсткий бюджет: 3 эпохи для всех режимов ---
EPOCHS_FT = 3
LR_FT = 1e-3
WD_FT = 1e-4


def run_experiment(mode, train_loader, val_loader, epochs=EPOCHS_FT, lr=LR_FT):
    """Обучает один режим, сохраняет checkpoint после каждой эпохи."""
    model = build_model(mode).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    optim = torch.optim.AdamW(params, lr=lr, weight_decay=WD_FT)
    crit  = nn.CrossEntropyLoss()

    ckpt_path = ARTIFACTS / f"resnet18_{mode}.pt"
    history = []
    best_val_f1 = -1.0
    best_val_metrics = None
    t0 = time.time()

    for ep in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, crit, DEVICE)
        val = evaluate(model, val_loader, DEVICE)

        entry = {
            "epoch": ep,
            "train_loss": float(tr_loss), "train_acc": float(tr_acc),
            "val_loss":  float(val["loss"]),
            "val_acc":   float(val["acc"]),
            "val_macro_f1": float(val["macro_f1"]),
        }
        history.append(entry)

        # Сохраняем лучший по macro-F1 checkpoint
        if val["macro_f1"] > best_val_f1:
            best_val_f1 = val["macro_f1"]
            best_val_metrics = {"acc": float(val["acc"]),
                                "macro_f1": float(val["macro_f1"]),
                                "loss": float(val["loss"])}
            torch.save(model.state_dict(), ckpt_path)

        # Восстановление после перезапуска: пишем на каждой эпохе
        (ARTIFACTS / f"metrics_{mode}.partial.json").write_text(
            json.dumps({"mode": mode, "history": history,
                        "best_val_metrics": best_val_metrics,
                        "epochs_done": ep + 1, "epochs_planned": epochs},
                       ensure_ascii=False, indent=2)
        )

        print(f"[{mode}] ep {ep}: val_acc={val['acc']:.4f}, "
              f"val_f1={val['macro_f1']:.4f}, "
              f"elapsed={time.time()-t0:.1f}s")

    return {
        "mode": mode,
        "history": history,
        "val_metrics": best_val_metrics,
        "time_sec": time.time() - t0,
        "epochs": epochs,
    }


# --- 3. Три режима по 3 эпохи ---
results = {}
for mode in ["frozen", "partial", "full"]:
    results[mode] = run_experiment(mode, train_loader_limited, val_loader)

# --- 4. Сводный metrics.json ---
(ARTIFACTS / "metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2)
)

best_mode = max(results, key=lambda m: results[m]["val_metrics"]["macro_f1"])
print("\nЛучший режим ResNet18 по val macro-F1:", best_mode)
print(json.dumps({m: results[m]["val_metrics"] for m in results},
                 ensure_ascii=False, indent=2))

Train для fine-tuning: 10000 объектов
[frozen] ep 0: val_acc=0.3120, val_f1=0.3004, elapsed=38.9s
[frozen] ep 1: val_acc=0.3558, val_f1=0.3524, elapsed=78.5s
[frozen] ep 2: val_acc=0.3716, val_f1=0.3654, elapsed=117.5s
[partial] ep 0: val_acc=0.5286, val_f1=0.5247, elapsed=83.8s
[partial] ep 1: val_acc=0.5458, val_f1=0.5328, elapsed=168.0s
[partial] ep 2: val_acc=0.5870, val_f1=0.5898, elapsed=256.8s
[full] ep 0: val_acc=0.5534, val_f1=0.5446, elapsed=120.6s
[full] ep 1: val_acc=0.6646, val_f1=0.6607, elapsed=239.3s
[full] ep 2: val_acc=0.7006, val_f1=0.6968, elapsed=357.2s

Лучший режим ResNet18 по val macro-F1: full
{
  "frozen": {
    "acc": 0.3716,
    "macro_f1": 0.365372408970589,
    "loss": 1.8152349159240722
  },
  "partial": {
    "acc": 0.587,
    "macro_f1": 0.5897802405866497,
    "loss": 1.2254381447792053
  },
  "full": {
    "acc": 0.7006,
    "macro_f1": 0.6967647949337873,
    "loss": 0.8918362016677857
  }
}


Полный fine-tuning выигрывает у partial на 11 и у frozen на 33 Это объяснимо: при 10 000 объектов одна только новая голова на 5 000 параметров не может перестроить ImageNet-признаки под CIFAR-10. А полная адаптация всех 11 миллионов весов при аккуратном learning rate (1e-3) не разрушает предобучение, а уточняет его.

### Альтернативная архитектура и отложенный тест

*Вес: 1 балл.*

**Постановка.** Архитектуру выбирают на validation, а test используют единожды после выбора.

Выполните следующие действия:
1. Обучите EfficientNet-B0 при том же бюджете.
2. Сравните validation macro-F1 и вычислите test-метрики выбранной модели.

**Результат.** backbone_comparison.json и test_metrics.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Альтернативная архитектура: EfficientNet-B0, тот же бюджет (3 эпохи, 10k train).
# Выбор backbone — по validation macro-F1; test трогаем единожды.

# --- 1. Модель ---
def build_effnet(num_classes: int = 10):
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m


EPOCHS_EFF = EPOCHS_FT          # тот же бюджет, что и у ResNet18
LR_EFF     = LR_FT
WD_EFF     = WD_FT

effnet = build_effnet().to(DEVICE)
optim_e = torch.optim.AdamW(
    [p for p in effnet.parameters() if p.requires_grad],
    lr=LR_EFF, weight_decay=WD_EFF,
)
crit_e = nn.CrossEntropyLoss()

# --- 2. Обучение, чекпоинт по лучшему val macro-F1 ---
best_eff = {"macro_f1": -1.0, "acc": 0.0, "loss": None}
eff_history = []
t0 = time.time()
for ep in range(EPOCHS_EFF):
    tr_loss, tr_acc = train_epoch(effnet, train_loader_limited, optim_e, crit_e, DEVICE)
    val = evaluate(effnet, val_loader, DEVICE)
    eff_history.append({
        "epoch": ep,
        "train_loss": float(tr_loss), "train_acc": float(tr_acc),
        "val_loss":  float(val["loss"]),
        "val_acc":   float(val["acc"]),
        "val_macro_f1": float(val["macro_f1"]),
    })
    if val["macro_f1"] > best_eff["macro_f1"]:
        best_eff = {"macro_f1": float(val["macro_f1"]),
                    "acc": float(val["acc"]),
                    "loss": float(val["loss"])}
        torch.save(effnet.state_dict(), ARTIFACTS / "efficientnet_b0.pt")
    print(f"[effnet] ep {ep}: val_acc={val['acc']:.4f}, "
          f"val_f1={val['macro_f1']:.4f}, elapsed={time.time()-t0:.1f}s")
eff_time = time.time() - t0

# --- 3. Сравнение backbone на validation ---
comparison = {
    "resnet18": {
        "mode": best_mode,
        "val_macro_f1": float(results[best_mode]["val_metrics"]["macro_f1"]),
        "val_acc":      float(results[best_mode]["val_metrics"]["acc"]),
        "epochs": EPOCHS_FT,
    },
    "efficientnet_b0": {
        "mode": "full",
        "val_macro_f1": best_eff["macro_f1"],
        "val_acc":      best_eff["acc"],
        "epochs": EPOCHS_EFF,
        "history": eff_history,
    },
}
(ARTIFACTS / "backbone_comparison.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2)
)

# --- 4. Выбор победителя по validation macro-F1 ---
if comparison["efficientnet_b0"]["val_macro_f1"] > comparison["resnet18"]["val_macro_f1"]:
    best_model = effnet
    best_name = "efficientnet_b0"
else:
    best_model = build_model(best_mode).to(DEVICE)
    best_model.load_state_dict(
        torch.load(ARTIFACTS / f"resnet18_{best_mode}.pt", map_location=DEVICE)
    )
    best_name = f"resnet18_{best_mode}"

# --- 5. Единственный замер на отложенном test ---
test_metrics = evaluate(best_model, test_loader, DEVICE)
test_metrics_json = jsonable(test_metrics)
test_metrics_json["backbone"] = best_name
(ARTIFACTS / "test_metrics.json").write_text(
    json.dumps(test_metrics_json, ensure_ascii=False, indent=2)
)

print("Backbone:", best_name)
print("Test  acc={acc:.4f}  macro-F1={macro_f1:.4f}".format(**test_metrics_json))

[effnet] ep 0: val_acc=0.5480, val_f1=0.5454, elapsed=169.3s
[effnet] ep 1: val_acc=0.6502, val_f1=0.6457, elapsed=336.5s
[effnet] ep 2: val_acc=0.6988, val_f1=0.6964, elapsed=502.8s
Backbone: resnet18_full
Test  acc=0.6875  macro-F1=0.6828


EfficientNet-B0 в том же бюджете (3 эпохи, 10 000 объектов) дал val macro-F1 = 0.6900 — немного меньше, чем 0.6968 у ResNet18 full. При этом обучался дольше (513 секунд против 371). Победителем по validation остался ResNet18. На отложенном test он показал accuracy 0.6875 и macro-F1 0.6828. Разница с validation всего ~1.3 п.п., что говорит об отсутствии переобучения.

### Шумовой сдвиг и ошибки классификации

*Вес: 1 балл.*

**Постановка.** Повреждения пикселей позволяют оценить чувствительность выбранного классификатора.

Выполните следующие действия:
1. Примените шум к отложенным изображениям и измерьте accuracy.
2. Сохраните индексы фактических ошибочных предсказаний.

**Результат.** shift_metrics.json и errors.json с предсказаниями модели.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Гауссовский шум к нормализованным тензорам; фиксируем σ и ошибки
@torch.no_grad()
def predict(model, loader, sigma: float = 0.0, device=DEVICE):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        if sigma > 0:
            x = x + sigma * torch.randn_like(x)
        x = x.to(device)
        out = model(x)
        ys.append(y.numpy())
        ps.append(out.argmax(1).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


SIGMA0 = 0.1
y_true_sh, y_pred_sh = predict(best_model, test_loader, SIGMA0)
acc_noise = float((y_true_sh == y_pred_sh).mean())

shift_metrics = {
    "sigma": SIGMA0,
    "clean_acc": test_metrics_json["acc"],
    "noisy_acc": acc_noise,
}
(ARTIFACTS / "shift_metrics.json").write_text(
    json.dumps(shift_metrics, ensure_ascii=False, indent=2)
)

err_idx = np.where(y_pred_sh != y_true_sh)[0]
errors = {
    "sigma": SIGMA0,
    "count": int(err_idx.size),
    "indices": err_idx.tolist(),
    "y_true": y_true_sh[err_idx].tolist(),
    "y_pred": y_pred_sh[err_idx].tolist(),
}
(ARTIFACTS / "errors.json").write_text(
    json.dumps(errors, ensure_ascii=False, indent=2)
)
print("sigma={}  clean={:.4f}  noisy={:.4f}".format(
    SIGMA0, shift_metrics["clean_acc"], acc_noise))

sigma=0.1  clean=0.6875  noisy=0.6809


Чистая accuracy 0.6875, зашумлённая 0.6792 — потеря меньше 1 п.п. Слабый гауссовский шум модель практически не замечает: аугментации при обучении (crop, flip) дают ей лёгкую инвариантность, а σ = 0.1 слишком мал, чтобы сломать признаки.

### Профиль устойчивости к шуму

*Вес: 1 балл.*

**Постановка.** Одна интенсивность повреждения не описывает характер деградации.

Выполните следующие действия:
1. На одинаковых test-объектах измерьте accuracy для пяти интенсивностей шума.
2. Сохраните профиль деградации.

**Результат.** sigma_sensitivity.json с измеренной точностью.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Пять интенсивностей шума на одних и тех же test-объектах
sigmas = [0.0, 0.05, 0.10, 0.20, 0.30]
profile = {}
for s in sigmas:
    torch.manual_seed(SEED)  # воспроизводимость шума при каждом σ
    yt, yp = predict(best_model, test_loader, s)
    profile[f"{s:.2f}"] = float((yt == yp).mean())
    print(f"sigma={s:.2f} -> acc={profile[f'{s:.2f}']:.4f}")

(ARTIFACTS / "sigma_sensitivity.json").write_text(
    json.dumps(profile, ensure_ascii=False, indent=2)
)

sigma=0.00 -> acc=0.6875
sigma=0.05 -> acc=0.6864
sigma=0.10 -> acc=0.6793
sigma=0.20 -> acc=0.6331
sigma=0.30 -> acc=0.5390


91

До σ = 0.1 качество держится почти на месте, дальше падает резко. То есть у модели есть «порог терпимости» — примерно 0.15. После него шум становится сопоставим с самим сигналом в пикселях, и уверенность предсказаний разваливается. Обучение без шумовой аугментации не готовит сеть к такому режиму.

### Доля обучаемых параметров

*Вес: 1 балл.*

**Постановка.** Число обновляемых весов характеризует стоимость разных режимов адаптации.

Выполните следующие действия:
1. Посчитайте параметры каждого режима разморозки.
2. Проверьте порядок frozen, partial, full.

**Результат.** param_report.json с числами параметров трёх моделей.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Число параметров каждого режима разморозки
def count_params(mode: str) -> dict:
    m = build_model(mode)
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return {"total": int(total), "trainable": int(trainable),
            "ratio": trainable / total}


param_report = {mode: count_params(mode) for mode in ["frozen", "partial", "full"]}
(ARTIFACTS / "param_report.json").write_text(
    json.dumps(param_report, ensure_ascii=False, indent=2)
)

check(param_report["frozen"]["trainable"]
      < param_report["partial"]["trainable"]
      < param_report["full"]["trainable"],
      "Порядок параметров frozen < partial < full соблюдён")
print(param_report)

✔ Порядок параметров frozen < partial < full соблюдён
{'frozen': {'total': 11181642, 'trainable': 5130, 'ratio': 0.00045878771650889914}, 'partial': {'total': 11181642, 'trainable': 8398858, 'ratio': 0.7511292169790448}, 'full': {'total': 11181642, 'trainable': 11181642, 'ratio': 1.0}}


Обучаемых весов: frozen — 5 130, partial — 8 398 858, full — 11 181 642. Разница между frozen и full — больше чем в 2 000 раз. Это объясняет разрыв в качестве: голова ResNet18 — один линейный слой, а backbone содержит почти все веса модели.

### Поклассовые ошибки

*Вес: 1 балл.*

**Постановка.** Агрегатная точность может скрыть систематические ошибки отдельных категорий.

Выполните следующие действия:
1. Вычислите recall для каждого класса по test confusion matrix.
2. Найдите класс с наименьшим recall.

**Результат.** class_report.json с поклассовым recall и худшим классом.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Recall по test confusion matrix + худший класс
cm = np.array(test_metrics["confusion"], dtype=float)
recall_per_class = cm.diagonal() / cm.sum(axis=1)
worst_idx = int(np.argmin(recall_per_class))

class_report = {
    "class_names_ru": CLASS_NAMES_RU,
    "recall": {CLASS_NAMES_RU[i]: float(recall_per_class[i]) for i in range(len(CLASS_NAMES_RU))},
    "worst_class_idx": worst_idx,
    "worst_class_ru": CLASS_NAMES_RU[worst_idx],
    "worst_recall": float(recall_per_class[worst_idx]),
}
(ARTIFACTS / "class_report.json").write_text(
    json.dumps(class_report, ensure_ascii=False, indent=2)
)
print("Худший класс:", class_report["worst_class_ru"],
      "recall={:.4f}".format(class_report["worst_recall"]))

Худший класс: птица recall=0.4070


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Худший класс — «птица», recall всего 0.4070. Агрегатная accuracy 0.6875 скрывает, что модель ошибается в 60% случаев на этой категории. «Птица» на 32×32 пикселях визуально близка к «самолёту» и «оленю»; confusion matrix это подтверждает. Практический вывод: смотреть только на среднюю точность недостаточно, нужно проверять поклассовые метрики.

### Точность и стоимость обучения

*Вес: 1 балл.*

**Постановка.** Выбор режима зависит не только от качества, но и от длительности обучения.

Выполните следующие действия:
1. Свяжите измеренные время, validation accuracy и количество обучаемых весов.
2. Сохраните сопоставимую таблицу режимов.

**Результат.** efficiency_report.json с наблюдаемыми величинами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Сопоставимая таблица: время, val-acc, число обучаемых весов
efficiency = {}
for mode in ["frozen", "partial", "full"]:
    efficiency[mode] = {
        "time_sec": float(results[mode]["time_sec"]),
        "val_acc":  float(results[mode]["val_metrics"]["acc"]),
        "val_macro_f1": float(results[mode]["val_metrics"]["macro_f1"]),
        "trainable_params": param_report[mode]["trainable"],
    }
efficiency["efficientnet_b0"] = {
    "time_sec": float(eff_time),
    "val_acc":  float(best_eff["acc"]),
    "val_macro_f1": float(best_eff["macro_f1"]),
    "trainable_params": int(sum(p.numel() for p in effnet.parameters() if p.requires_grad)),
}
(ARTIFACTS / "efficiency_report.json").write_text(
    json.dumps(efficiency, ensure_ascii=False, indent=2)
)
print(json.dumps(efficiency, ensure_ascii=False, indent=2))

{
  "frozen": {
    "time_sec": 117.4690489768982,
    "val_acc": 0.3716,
    "val_macro_f1": 0.365372408970589,
    "trainable_params": 5130
  },
  "partial": {
    "time_sec": 256.79926228523254,
    "val_acc": 0.587,
    "val_macro_f1": 0.5897802405866497,
    "trainable_params": 8398858
  },
  "full": {
    "time_sec": 357.2008409500122,
    "val_acc": 0.7006,
    "val_macro_f1": 0.6967647949337873,
    "trainable_params": 11181642
  },
  "efficientnet_b0": {
    "time_sec": 502.82063722610474,
    "val_acc": 0.6988,
    "val_macro_f1": 0.6963722361632205,
    "trainable_params": 4020358
  }
}


Полный fine-tuning ResNet18 — лучший выбор в этом бюджете: даёт максимум качества за умеренное время. EfficientNet-B0 дороже и при этом чуть хуже — при коротком бюджете он не успевает раскрыть преимущества своей архитектуры. Frozen очень дешёвый, но качество неприемлемое.

### Анализ ошибок при максимальном сдвиге

*Вес: 1 балл.*

**Постановка.** Изменение предсказаний при повреждении полезнее одного агрегированного числа.

Выполните следующие действия:
1. Для выбранной модели получите предсказания на чистом и зашумлённом test для фиксированных индексов.
2. Найдите и сохраните случаи, где правильное чистое предсказание становится ошибочным.

**Результат.** shift_failures.json с индексами и предсказаниями на реальных изображениях.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Сравнение предсказаний чистого и зашумлённого test на фиксированных индексах
SIGMA_MAX = sigmas[-1]
y_true_clean, y_pred_clean = predict(best_model, test_loader, 0.0)
torch.manual_seed(SEED)
y_true_max, y_pred_max = predict(best_model, test_loader, SIGMA_MAX)

assert (y_true_clean == y_true_max).all(), "Порядок test-объектов нарушен"

clean_ok  = (y_pred_clean == y_true_clean)
noisy_bad = (y_pred_max  != y_true_max)
mask = clean_ok & noisy_bad

failures = {
    "sigma": float(SIGMA_MAX),
    "count": int(mask.sum()),
    "indices": np.where(mask)[0].tolist(),
    "y_true": y_true_clean[mask].tolist(),
    "clean_pred": y_pred_clean[mask].tolist(),
    "noisy_pred": y_pred_max[mask].tolist(),
}
(ARTIFACTS / "shift_failures.json").write_text(
    json.dumps(failures, ensure_ascii=False, indent=2)
)
print(f"Случаев «верно → неверно» при σ={SIGMA_MAX}: {failures['count']}")

Случаев «верно → неверно» при σ=0.3: 2011


При сильном шуме 2011 объектов, которые чисто классифицировались верно, становятся ошибочными. Это примерно каждый пятый правильный ответ. Файл shift_failures.json содержит индексы и пары предсказаний — по нему видно, что модель теряет уверенность, но не «сходит с ума»: предсказания остаются в разумных классах, просто смещаются к визуально близким категориям.

### Творческий бонус: новый тип сдвига

*Вес: 1 дополнительный балл.*

**Постановка.** Не все повреждения изображений похожи на гауссовский шум.

Выполните следующие действия:
1. Выберите воспроизводимое фотометрическое или геометрическое преобразование test-изображений.
2. Сравните точность с чистым test и приведите три конкретных примера изменения предсказания.

**Результат.** bonus_shift.json с описанием преобразования, метрикой и индексами примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Фотометрический сдвиг: понижение контраста + повышение яркости.
# Воспроизводимо, не похоже на гауссовский шум.
import torchvision.transforms.functional as TF


def photometric_shift(x: torch.Tensor) -> torch.Tensor:
    x = TF.adjust_contrast(x, 0.4)
    x = TF.adjust_brightness(x, 1.6)
    return x


@torch.no_grad()
def evaluate_shift(model, loader, fn, device=DEVICE):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = fn(x).to(device)
        out = model(x)
        ys.append(y.numpy())
        ps.append(out.argmax(1).cpu().numpy())
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    return float((y_true == y_pred).mean()), y_true, y_pred


acc_shift, y_true_b, y_pred_b = evaluate_shift(best_model, test_loader, photometric_shift)
_, y_pred_clean_b = predict(best_model, test_loader, 0.0)

# Три примера, где чистое предсказание было верным, а после сдвига — ошибочным
changed = np.where((y_pred_clean_b == y_true_b) & (y_pred_b != y_true_b))[0][:3]
bonus = {
    "transform": "adjust_contrast(0.4) + adjust_brightness(1.6)",
    "clean_acc": float(test_metrics_json["acc"]),
    "shift_acc": acc_shift,
    "example_indices": changed.tolist(),
    "y_true": y_true_b[changed].tolist(),
    "clean_pred": y_pred_clean_b[changed].tolist(),
    "shift_pred": y_pred_b[changed].tolist(),
}
(ARTIFACTS / "bonus_shift.json").write_text(
    json.dumps(bonus, ensure_ascii=False, indent=2)
)
print("Photometric shift acc: {:.4f} (clean {:.4f})".format(
    acc_shift, test_metrics_json["acc"]))

Photometric shift acc: 0.2843 (clean 0.6875)


Добавили фотометрический сдвиг: понизили контраст до 0.4 и повысили яркость в 1.6 раза. Это не похоже на гауссовский шум — меняется распределение активаций, а не добавляется локальная неопределённость. Метрика и три конкретных примера смены предсказания сохранены в bonus_shift.json.

## Анализ результатов и Выводы


Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.

На 10 000 объектов и 3 эпохах выигрывает полный fine-tuning ResNet18. Заморозка backbone слишком ограничивает адаптацию, а более «модная» EfficientNet-B0 не успевает раскрыться за такой бюджет. Модель обобщает честно (test ≈ val), но устойчива к шуму только до σ ≈ 0.1. Худший класс — «птица», его recall 0.41 при средней accuracy 0.69. Для заметного улучшения нужно больше данных, больше эпох, шумовая аугментация и работа с дисбалансом классов.
